# Lecture 7 — UCS mechanical-data reduction

## Part 1: from TDMS measurements to stress and strain

This version deliberately stops after the first reduction and quality-control plot. The workflow is:

1. inspect the file metadata, PID settings and calibrations;
2. extract only the mechanical time series from the `ADC` group;
3. explore any pair of recorded variables in a pop-up window;
4. calculate stress, corrected displacement and strain over the complete experiment;
5. explore the reduced variables with the same pop-up tool.

The original TDMS file remains unchanged. Recorded, calibrated and calculated quantities are kept conceptually separate.

> **Sign convention:** compressive stress and axial shortening strain are positive.

## 0. Environment

The pop-up explorer uses Matplotlib's Qt backend. In the Anaconda environment used for the course, install the packages once if needed:

```bash
conda install -c conda-forge nptdms numpy pandas matplotlib pyqt
```

Restart the Jupyter kernel after installing `pyqt`.

In [3]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from nptdms import TdmsFile
import matplotlib.pyplot as plt
from matplotlib.widgets import RadioButtons, Button, TextBox
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

COLORS = {
    'ink': '#18323F',
    'teal': '#087E8B',
    'orange': '#D99539',
    'gray': '#52636B',
    'light': '#AFC4C9',
}


## 1. Select the TDMS file

Put the notebook and TDMS file in the same folder, or change the path below. The `.tdms_index` sidecar may remain beside the file, but `nptdms` reads the `.tdms` file directly.

In [4]:
TDMS_PATH = Path('triax_calibration.tdms')

if not TDMS_PATH.exists():
    raise FileNotFoundError(
        f'{TDMS_PATH.resolve()} was not found. Put the TDMS file beside the notebook '
        'or edit TDMS_PATH.'
    )

print('Input file:', TDMS_PATH.resolve())


Input file: /Users/marcoscuderi/Library/CloudStorage/GoogleDrive-marco.scuderi@uniroma1.it/My Drive/Comunications/Classes/04_petrophysics/L07_part_a_b/UCS/calibration/triax_calibration.tdms


## 2. Import metadata, settings and ADC time series

The TDMS file contains three different kinds of information, and they should not be mixed:

- **file properties:** experiment identity and startup information;
- **starting PID settings:** controller parameters recorded before the experiment;
- **ADC group:** time series of the measured mechanical quantities.

In the supplied file, PID values are stored in the file-level `Starting PID Settings` property. Calibration slopes and intercepts are stored as properties of individual ADC channels. The following cells unpack and display them separately.

In [5]:
tdms = TdmsFile.read(TDMS_PATH)

file_properties = dict(tdms.properties)
general_metadata = {
    key: value for key, value in file_properties.items()
    if 'PID' not in key.upper() and 'CALIB' not in key.upper()
}

display(Markdown('### Experiment metadata'))
display(pd.DataFrame(general_metadata.items(), columns=['property', 'value']))


### Experiment metadata

,property,value
0,name,triax_calibration
1,Experiment_Name,triax_calibration
2,Operator,scuderi
3,Manufacturer,Leeman Geophysical LLC
4,Apparatus,biax2
5,Start_Datetime,"Friday, October 9, 2026 10:13:56 AM"
6,Software_Version,mar042026


In [6]:
def parse_pid_settings(settings_text):
    records = []
    if not settings_text:
        return pd.DataFrame(columns=['controller', 'P', 'I', 'D'])

    pattern = re.compile(r'^(.*?)\s+([PID]):\s*([-+0-9.eE]+)\s*$')
    for line in str(settings_text).splitlines():
        line = line.strip()
        if not line:
            continue
        match = pattern.match(line)
        if match:
            controller, term, value = match.groups()
            records.append({
                'controller': controller.strip(),
                'term': term,
                'value': float(value),
            })
        else:
            records.append({'controller': line, 'term': 'unparsed', 'value': np.nan})

    long_table = pd.DataFrame(records)
    parsed = long_table[long_table['term'].isin(['P', 'I', 'D'])]
    if parsed.empty:
        return long_table
    return (parsed.pivot(index='controller', columns='term', values='value')
            .reindex(columns=['P', 'I', 'D']).reset_index())


pid_text = file_properties.get('Starting PID Settings', '')
pid_settings = parse_pid_settings(pid_text)

display(Markdown('### Starting PID settings'))
display(pid_settings)


### Starting PID settings

term,controller,P,I,D
0,Horizontal Displacement,1.0,0.0,0.0
1,Horizontal Load,0.0,0.0,0.0
2,PPa Displacement,1.0,0.0,0.0
3,PPa Load,0.0,0.0,0.0
4,PPb Displacement,1.0,0.0,0.0
5,PPb Load,0.0,0.0,0.0
6,Pc Displacement,1.0,0.0,0.0
7,Pc Load,0.0,0.0,0.0
8,Vertical Displacement,1.0,0.0,0.0
9,Vertical Load,0.0,0.0,0.0


### Experiment comments and associated ADC rows

Comments are stored in the TDMS `Metadata` group as paired `Comment Time` and `Comment` channels. The code below associates each comment with the nearest row of the ADC time series. This makes laboratory notes directly traceable to the mechanical record.

If the file contains no comments, an empty table is shown rather than causing an error.

In [7]:
def find_channel_case_insensitive(group, requested_name):
    matches = {
        channel.name.strip().casefold(): channel
        for channel in group.channels()
    }
    return matches.get(requested_name.strip().casefold())


def extract_comments_with_rows(tdms_file, adc_group_name='ADC', metadata_group_name='Metadata'):
    columns = ['comment_number', 'ADC_row', 'comment_time', 'ADC_time',
               'time_difference_s', 'comment']
    group_names = {group.name: group for group in tdms_file.groups()}
    if metadata_group_name not in group_names or adc_group_name not in group_names:
        return pd.DataFrame(columns=columns)

    metadata_group = group_names[metadata_group_name]
    adc_group_local = group_names[adc_group_name]
    comment_time_channel = find_channel_case_insensitive(metadata_group, 'Comment Time')
    comment_channel = find_channel_case_insensitive(metadata_group, 'Comment')
    adc_time_channel = find_channel_case_insensitive(adc_group_local, 'Time')

    if comment_time_channel is None or comment_channel is None:
        return pd.DataFrame(columns=columns)

    comment_times = np.asarray(comment_time_channel[:], dtype=float)
    comments = np.asarray(comment_channel[:]).astype(str)
    if len(comment_times) != len(comments):
        raise ValueError('Comment Time and Comment channels have different lengths.')

    if adc_time_channel is None or len(adc_time_channel) == 0:
        return pd.DataFrame({
            'comment_number': np.arange(len(comments), dtype=int),
            'ADC_row': pd.array([pd.NA] * len(comments), dtype='Int64'),
            'comment_time': comment_times,
            'ADC_time': np.nan,
            'time_difference_s': np.nan,
            'comment': comments,
        })[columns]

    adc_times = np.asarray(adc_time_channel[:], dtype=float)
    insertion_points = np.searchsorted(adc_times, comment_times)
    nearest_rows = []
    for comment_time, insertion in zip(comment_times, insertion_points):
        candidates = [i for i in (insertion - 1, insertion) if 0 <= i < len(adc_times)]
        nearest_rows.append(min(candidates, key=lambda i: abs(adc_times[i] - comment_time)))

    nearest_rows = np.asarray(nearest_rows, dtype=int)
    nearest_times = adc_times[nearest_rows]
    return pd.DataFrame({
        'comment_number': np.arange(len(comments), dtype=int),
        'ADC_row': nearest_rows,
        'comment_time': comment_times,
        'ADC_time': nearest_times,
        'time_difference_s': comment_times - nearest_times,
        'comment': comments,
    })[columns]


comments_table = extract_comments_with_rows(tdms)
display(Markdown('### Recorded comments'))
if comments_table.empty:
    print('No comments were recorded in this TDMS file.')
else:
    display(comments_table)


### Recorded comments

No comments were recorded in this TDMS file.


### Discover the ADC channels and assign units

The notebook does not assume a fixed acquisition layout. It reads every channel found in the TDMS `ADC` group and reports its recorded name and number of samples.

Run the discovery cell first. In the following cell, enter **only the units**, in the same order as the displayed channels. Do not type or maintain a separate channel-name list.

In [8]:
ADC_GROUP_NAME = 'ADC'
if ADC_GROUP_NAME not in [group.name for group in tdms.groups()]:
    raise KeyError(f'TDMS group {ADC_GROUP_NAME!r} was not found.')

adc_group = tdms[ADC_GROUP_NAME]
ADC_CHANNEL_NAMES = [channel.name for channel in adc_group.channels()]

adc_discovery = pd.DataFrame({
    'order': np.arange(1, len(ADC_CHANNEL_NAMES) + 1, dtype=int),
    'channel': ADC_CHANNEL_NAMES,
    'records': [len(channel) for channel in adc_group.channels()],
})

display(Markdown('### Channels found in the ADC group'))
display(adc_discovery)
print(f'{len(ADC_CHANNEL_NAMES)} ADC channels discovered automatically.')


### Channels found in the ADC group

,order,channel,records
0,1,Vertical Load,10740
1,2,Horizontal Load,10740
2,3,Vertical Displacement,10740
3,4,Horizontal Displacement,10740
4,5,AUX 5,10740
5,6,Time,10740


6 ADC channels discovered automatically.


In [9]:
# Enter one unit for each channel shown above, preserving the displayed order.
# Only this list needs to be edited when opening a different experiment.
USER_ADC_UNITS = [
    'kN',  # 1
    'kN',  # 2
    'µm',  # 3
    'µm',  # 4
    'V',   # 5
    's',   # 6
]

if len(USER_ADC_UNITS) != len(ADC_CHANNEL_NAMES):
    raise ValueError(
        f'Enter exactly {len(ADC_CHANNEL_NAMES)} units in USER_ADC_UNITS; '
        f'{len(USER_ADC_UNITS)} were supplied.'
    )

ADC_UNITS = dict(zip(ADC_CHANNEL_NAMES, USER_ADC_UNITS))
unit_assignment = adc_discovery.copy()
unit_assignment['unit'] = [ADC_UNITS[name] for name in ADC_CHANNEL_NAMES]
display(Markdown('### User-confirmed ADC units'))
display(unit_assignment)


### User-confirmed ADC units

,order,channel,records,unit
0,1,Vertical Load,10740,kN
1,2,Horizontal Load,10740,kN
2,3,Vertical Displacement,10740,µm
3,4,Horizontal Displacement,10740,µm
4,5,AUX 5,10740,V
5,6,Time,10740,s


In [10]:
def build_calibration_table(group, unit_map):
    rows = []
    for channel in group.channels():
        base_name = (channel.name.strip().lower().replace(' ', '_').replace('-', '_')
                     .replace('/', '_').replace('(', '').replace(')', ''))
        slope = float(channel.properties.get('Slope', 1.0))
        intercept = float(channel.properties.get('Intercept', 0.0))
        rows.append({
            'channel': channel.name,
            'raw_column': f'{base_name}_raw',
            'calibrated_column': f'{base_name}_calibrated',
            'slope_m': slope,
            'intercept_b': intercept,
            'engineering_unit': unit_map.get(channel.name, 'VERIFY'),
            'calculation': (
                f'{base_name}_calibrated = {slope:.8g} × '
                f'{base_name}_raw + ({intercept:.8g})'
            ),
        })
    return pd.DataFrame(rows)


calibration_table = build_calibration_table(adc_group, ADC_UNITS)
display(Markdown('### ADC calibration coefficients'))
display(calibration_table)


### ADC calibration coefficients

,channel,raw_column,calibrated_column,slope_m,intercept_b,engineering_unit,calculation
0,Vertical Load,vertical_load_raw,vertical_load_calibrated,23.203,-1.843994,kN,vertical_load_calibrated = 23.203 × vertical_l...
1,Horizontal Load,horizontal_load_raw,horizontal_load_calibrated,25.708,24.133717,kN,horizontal_load_calibrated = 25.708 × horizont...
2,Vertical Displacement,vertical_displacement_raw,vertical_displacement_calibrated,3662.900,29509.795097,µm,vertical_displacement_calibrated = 3662.9 × ve...
3,Horizontal Displacement,horizontal_displacement_raw,horizontal_displacement_calibrated,1613.800,3912.779970,µm,horizontal_displacement_calibrated = 1613.8 × ...
4,AUX 5,aux_5_raw,aux_5_calibrated,1.000,0.000000,V,aux_5_calibrated = 1 × aux_5_raw + (0)
5,Time,time_raw,time_calibrated,1.000,0.000000,s,time_calibrated = 1 × time_raw + (0)


The calibration convention follows the existing `rawPy.py` routine. For every ADC channel, the notebook retains exactly two versions:

- `*_raw`: the values stored in the TDMS ADC channel;
- `*_calibrated`: the values calculated in engineering units.

For every row (i):

$
x_{calibrated,i}=m\,x_{raw,i}+b
$

where (m) is the channel `Slope` property and (b) is its `Intercept` property. Calibrated loads are in kN, calibrated displacements in mm, and AUX channels in V. The table above reports the actual equation applied to each channel.

In [11]:
def safe_name(name):
    return (name.strip().lower().replace(' ', '_').replace('-', '_')
            .replace('/', '_').replace('(', '').replace(')', ''))


def extract_adc_data(group, unit_map):
    data = {}
    inventory_rows = []

    lengths = [len(channel) for channel in group.channels()]
    if len(set(lengths)) != 1:
        raise ValueError(f'ADC channels do not have equal record counts: {sorted(set(lengths))}')

    for channel in group.channels():
        raw_values = np.asarray(channel[:], dtype=float)
        slope = float(channel.properties.get('Slope', 1.0))
        intercept = float(channel.properties.get('Intercept', 0.0))
        name = safe_name(channel.name)

        # Retain the recorded values and one calibrated engineering-unit series.
        data[f'{name}_raw'] = raw_values
        data[f'{name}_calibrated'] = raw_values * slope + intercept

        inventory_rows.append({
            'channel': channel.name,
            'records': len(raw_values),
            'unit': unit_map.get(channel.name, 'VERIFY'),
        })

    frame = pd.DataFrame(data)
    frame.insert(0, 'record', np.arange(len(frame), dtype=int))
    inventory = pd.DataFrame(inventory_rows)
    return frame, inventory


adc_data, adc_inventory = extract_adc_data(adc_group, ADC_UNITS)

if 'time_calibrated' not in adc_data.columns:
    raise KeyError('The calibrated Time channel was not found in the ADC group.')
adc_data['elapsed_time_s'] = (
    adc_data['time_calibrated'] - adc_data['time_calibrated'].iloc[0]
)

display(Markdown('### ADC time-series inventory'))
display(adc_inventory)
print(f'Total mechanical records: {len(adc_data):,}')


### ADC time-series inventory

,channel,records,unit
0,Vertical Load,10740,kN
1,Horizontal Load,10740,kN
2,Vertical Displacement,10740,µm
3,Horizontal Displacement,10740,µm
4,AUX 5,10740,V
5,Time,10740,s


Total mechanical records: 10,740


## 3. Explore and, if needed, offset the recorded experiment

Use the pop-up to inspect the calibrated ADC channels. Hover over the curve to read the record number and coordinates.

If an electrical spike produces a step in a calibrated signal:

1. Select that calibrated channel as the **Y variable**.
2. Enter the **start row** immediately before the step and the **end row** immediately after it.
3. Press **Apply step offset** and review the proposed correction.
4. Confirm to align the end value with the start value and shift every subsequent record by the same amount.

The short interval between the two selected rows is replaced by a straight bridge, removing the spike transition. The confirmed change is made directly in `adc_data`, so Step 4 and the exported file use the corrected values. The corresponding `_raw` channel is never modified, and every accepted correction is recorded in `adc_offset_history`.

The operation is cumulative: repeat it as many times as necessary on the same channel. Each new correction uses the already corrected signal, which is useful for removing the upward and downward edges of the same electrical disturbance.


In [15]:
%matplotlib qt

def open_xy_explorer(
    data, units=None, title='Channel explorer', include_raw=False,
    enable_offset=False, offset_history=None,
):
    # Open an interactive X-Y explorer. When enable_offset is True, the
    # selected calibrated Y channel can have a permanent step correction made
    # in data. The end row is aligned to the start row and the same shift is
    # added to every subsequent record. Corrections can be repeated.
    # Raw ADC columns are never modified.
    units = units or {}
    offset_history = offset_history if offset_history is not None else []
    candidates = [
        column for column in data.columns
        if np.issubdtype(data[column].dtype, np.number)
        and (include_raw or not column.endswith('_raw'))
    ]
    if len(candidates) < 2:
        raise ValueError('At least two numeric columns are required.')

    # Only calibrated measured quantities can be offset. Record number,
    # elapsed time, calculated quantities, and raw ADC data are protected.
    offset_candidates = [
        column for column in candidates
        if column.endswith('_calibrated') and column != 'time_calibrated'
    ]

    preferred_x = 'elapsed_time_s' if 'elapsed_time_s' in candidates else candidates[0]
    preferred_y = (
        'vertical_load_calibrated'
        if 'vertical_load_calibrated' in candidates else candidates[1]
    )

    fig = plt.figure(figsize=(13.5, 7.8))
    fig.canvas.manager.set_window_title(title)
    bottom = 0.18 if enable_offset else 0.12
    plot_ax = fig.add_axes([0.31, bottom, 0.66, 0.92 - bottom])
    x_ax = fig.add_axes([0.02, 0.51, 0.25, 0.41], facecolor='#F2F6F6')
    y_ax = fig.add_axes([0.02, 0.08, 0.25, 0.37], facecolor='#F2F6F6')
    reset_ax = fig.add_axes([0.84, 0.025, 0.12, 0.055])

    x_selector = RadioButtons(x_ax, candidates, active=candidates.index(preferred_x))
    y_selector = RadioButtons(y_ax, candidates, active=candidates.index(preferred_y))
    reset_button = Button(reset_ax, 'Reset view')
    for selector_label in list(x_selector.labels) + list(y_selector.labels):
        selector_label.set_fontsize(8.5)
    x_ax.set_title('X variable', fontsize=11, loc='left', color=COLORS['ink'])
    y_ax.set_title('Y variable', fontsize=11, loc='left', color=COLORS['ink'])

    state = {'x': preferred_x, 'y': preferred_y}
    line, = plot_ax.plot(data[state['x']], data[state['y']], color=COLORS['teal'], lw=1.4)
    line.set_picker(6)
    annotation = plot_ax.annotate(
        '', xy=(0, 0), xytext=(15, 15), textcoords='offset points',
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                  edgecolor=COLORS['teal'], alpha=0.95),
        arrowprops=dict(arrowstyle='->', color=COLORS['teal'])
    )
    annotation.set_visible(False)

    offset_widgets = []
    status_text = None
    if enable_offset:
        start_ax = fig.add_axes([0.31, 0.045, 0.105, 0.047])
        end_ax = fig.add_axes([0.46, 0.045, 0.105, 0.047])
        apply_ax = fig.add_axes([0.61, 0.037, 0.18, 0.060])
        start_box = TextBox(start_ax, 'Start row ', initial='0')
        end_box = TextBox(end_ax, 'End row ', initial=str(min(20, len(data) - 1)))
        apply_button = Button(apply_ax, 'Apply step offset')
        status_text = fig.text(
            0.31, 0.115,
            'Step tool: select a calibrated Y channel and bracket one discontinuity.',
            fontsize=9, color=COLORS['gray'], va='center'
        )
        offset_widgets.extend([start_box, end_box, apply_button])

    def label(column):
        unit = units.get(column, '')
        return f'{column} [{unit}]' if unit else column

    def redraw():
        line.set_data(data[state['x']], data[state['y']])
        plot_ax.set_xlabel(label(state['x']))
        plot_ax.set_ylabel(label(state['y']))
        plot_ax.relim()
        plot_ax.autoscale_view()
        plot_ax.grid(alpha=0.22)
        plot_ax.set_title(f"{state['y']} versus {state['x']}", color=COLORS['ink'])
        annotation.set_visible(False)
        fig.canvas.draw_idle()

    def choose_x(value):
        state['x'] = value
        redraw()

    def choose_y(value):
        state['y'] = value
        if enable_offset and status_text is not None:
            if value in offset_candidates:
                status_text.set_text(
                    f'Step target: {value}. Enter rows immediately before and after one jump.'
                )
                status_text.set_color(COLORS['gray'])
            else:
                status_text.set_text(
                    'Protected quantity: choose an ADC channel ending in _calibrated.'
                )
                status_text.set_color('#A33A2B')
        redraw()

    def reset_view(_event):
        plot_ax.relim()
        plot_ax.autoscale_view()
        annotation.set_visible(False)
        fig.canvas.draw_idle()

    def show_coordinates(event):
        if event.inaxes != plot_ax:
            if annotation.get_visible():
                annotation.set_visible(False)
                fig.canvas.draw_idle()
            return

        contains, information = line.contains(event)
        if contains:
            point_index = information['ind'][0]
            x_value = line.get_xdata()[point_index]
            y_value = line.get_ydata()[point_index]
            annotation.xy = (x_value, y_value)
            annotation.set_text(
                f'row: {point_index:,}\n'
                f'x: {x_value:.6g}\n'
                f'y: {y_value:.6g}'
            )
            annotation.set_visible(True)
            fig.canvas.draw_idle()
        elif annotation.get_visible():
            annotation.set_visible(False)
            fig.canvas.draw_idle()

    def set_offset_status(message, color='#A33A2B'):
        if status_text is not None:
            status_text.set_text(message)
            status_text.set_color(color)
            fig.canvas.draw_idle()

    def apply_step_offset(_event):
        channel = state['y']
        if channel not in offset_candidates:
            set_offset_status(
                'Offset not applied: select a calibrated measured channel as Y.'
            )
            return

        try:
            start_value = float(start_box.text.strip())
            end_value = float(end_box.text.strip())
            if not start_value.is_integer() or not end_value.is_integer():
                raise ValueError
            start_row, end_row = int(start_value), int(end_value)
        except ValueError:
            set_offset_status('Offset not applied: start and end rows must be integers.')
            return

        if start_row < 0 or end_row >= len(data) or start_row > end_row:
            set_offset_status(
                f'Offset not applied: use 0 ≤ start ≤ end ≤ {len(data) - 1}.'
            )
            return

        series = pd.to_numeric(data[channel], errors='coerce').astype(float)
        start_value = float(series.iloc[start_row])
        end_value = float(series.iloc[end_row])
        if not np.isfinite(start_value) or not np.isfinite(end_value):
            set_offset_status('Offset not applied: start and end values must be finite.')
            return

        offset_added = start_value - end_value
        unit = units.get(channel, '')
        unit_text = f' {unit}' if unit else ''

        # Use Matplotlib's Qt compatibility layer so the confirmation works
        # with PyQt5, PyQt6, PySide2, or PySide6, whichever Matplotlib uses.
        from matplotlib.backends.qt_compat import QtWidgets
        message_box = QtWidgets.QMessageBox
        if hasattr(message_box, 'StandardButton'):
            yes_button = message_box.StandardButton.Yes
            no_button = message_box.StandardButton.No
        else:
            yes_button = message_box.Yes
            no_button = message_box.No

        message = (
            f'Channel: {channel}\n'
            f'Start row {start_row}: {start_value:.8g}{unit_text}\n'
            f'End row {end_row}: {end_value:.8g}{unit_text}\n'
            f'Offset added from row {end_row} onward: {offset_added:+.8g}{unit_text}\n\n'
            'Align the end point with the start point, bridge the selected interval,\n'
            'and apply the same shift to every following record?\n'
            'The corresponding raw ADC channel will remain unchanged.'
        )
        answer = message_box.question(
            fig.canvas.manager.window,
            'Confirm permanent offset',
            message,
            yes_button | no_button,
            no_button,
        )
        if answer != yes_button:
            set_offset_status('Offset cancelled; the data were not changed.', COLORS['gray'])
            return

        # Shift the end point and the complete following curve. A straight
        # bridge replaces any samples within the selected spike transition.
        corrected = series.copy()
        corrected.iloc[end_row:] = corrected.iloc[end_row:] + offset_added
        corrected_end_value = float(corrected.iloc[end_row])
        corrected.iloc[start_row:end_row + 1] = np.linspace(
            start_value, corrected_end_value, end_row - start_row + 1
        )

        # In-place assignment makes this cumulative correction available to
        # every later correction, Step 4, and the exported file.
        data.loc[:, channel] = corrected.to_numpy()
        offset_history.append({
            'correction_number': len(offset_history) + 1,
            'channel': channel,
            'start_row': start_row,
            'end_row': end_row,
            'start_value_before': start_value,
            'end_value_before': end_value,
            'offset_added_from_end': offset_added,
            'method': 'align end to start; shift all following records',
            'unit': unit,
        })
        set_offset_status(
            f'Applied #{len(offset_history)}: added {offset_added:+.6g}{unit_text} '
            f'from row {end_row} onward.',
            '#177245',
        )
        print(
            f'Permanent step correction #{len(offset_history)}: '
            f'{channel}, aligned row {end_row} to row {start_row}, '
            f'added {offset_added:+.8g}{unit_text} from row {end_row} onward. '
            'Raw ADC data unchanged.'
        )
        redraw()

    x_selector.on_clicked(choose_x)
    y_selector.on_clicked(choose_y)
    reset_button.on_clicked(reset_view)
    if enable_offset:
        apply_button.on_clicked(apply_step_offset)
    hover_connection = fig.canvas.mpl_connect('motion_notify_event', show_coordinates)
    redraw()
    plt.show(block=False)

    # Keep widget objects alive while the pop-up window is open.
    fig._xy_explorer_widgets = (
        x_selector, y_selector, reset_button, annotation, hover_connection,
        *offset_widgets,
    )
    return fig


RECORDED_UNITS = {'record': '#', 'elapsed_time_s': 's'}
for channel_name in ADC_CHANNEL_NAMES:
    base_name = safe_name(channel_name)
    RECORDED_UNITS[f'{base_name}_raw'] = 'raw ADC value'
    RECORDED_UNITS[f'{base_name}_calibrated'] = ADC_UNITS[channel_name]

# Accepted corrections remain available for inspection after the GUI is used.
adc_offset_history = []

raw_explorer = open_xy_explorer(
    adc_data,
    units=RECORDED_UNITS,
    title='UCS experiment — recorded ADC channels',
    include_raw=False,
    enable_offset=True,
    offset_history=adc_offset_history,
)


Permanent step correction #1: vertical_load_calibrated, aligned row 2862 to row 0, added +1.2204753 kN from row 2862 onward. Raw ADC data unchanged.
Permanent step correction #2: vertical_displacement_calibrated, aligned row 2862 to row 0, added +36.890649 µm from row 2862 onward. Raw ADC data unchanged.


## 4. Calculate stress, corrected displacement and strain

At this stage the experiment remains one continuous record. We do not yet separate individual tests.

### Analysis choices

- Choose the **calibrated** channels that represent axial load and axial displacement.
- Choose a zero record after inspecting the pop-up plot.
- Enter the specimen's measured initial diameter and length.
- Enter machine stiffness only if it has been measured independently.

The units are inherited from the ADC unit assignment rather than entered a second time. This prevents the same calibrated channel from being interpreted with conflicting units in different cells.

In [16]:
display(Markdown('### Confirmed Step 3 offset corrections'))
if adc_offset_history:
    display(pd.DataFrame(adc_offset_history))
else:
    print('No calibrated-channel offsets have been applied.')

ANALYSIS = {
    'load_channel': 'vertical_load_calibrated',
    'displacement_channel': 'vertical_displacement_calibrated',
    'zero_record': 2862,
    'zero_half_window_records': 5,
    'diameter_mm': 25,          # REQUIRED: measured specimen diameter
    'initial_length_mm': 50,    # REQUIRED: measured specimen length
    'shortening_sign': +1.0,        # +1 if recorded displacement increases during shortening
    'machine_stiffness_kN_per_mm': np.nan,  # optional independent calibration
}

display(pd.DataFrame(ANALYSIS.items(), columns=['parameter', 'value']))

selected_calibrations = calibration_table[
    calibration_table['calibrated_column'].isin([
        ANALYSIS['load_channel'], ANALYSIS['displacement_channel']
    ])
]
display(Markdown('### Calibration equations used by Step 4'))
display(selected_calibrations[[
    'channel', 'raw_column', 'calibrated_column', 'slope_m',
    'intercept_b', 'engineering_unit', 'calculation'
]])


### Confirmed Step 3 offset corrections

,correction_number,channel,start_row,end_row,start_value_before,end_value_before,offset_added_from_end,method,unit
0,1,vertical_load_calibrated,0,2862,1.322316,0.101841,1.220475,align end to start; shift all following records,kN
1,2,vertical_displacement_calibrated,0,2862,17030.796241,16993.905592,36.890649,align end to start; shift all following records,µm


,parameter,value
0,load_channel,vertical_load_calibrated
1,displacement_channel,vertical_displacement_calibrated
2,zero_record,2862
3,zero_half_window_records,5
4,diameter_mm,25
5,initial_length_mm,50
6,shortening_sign,1.0
7,machine_stiffness_kN_per_mm,NaN


### Calibration equations used by Step 4

,channel,raw_column,calibrated_column,slope_m,intercept_b,engineering_unit,calculation
0,Vertical Load,vertical_load_raw,vertical_load_calibrated,23.203,-1.843994,kN,vertical_load_calibrated = 23.203 × vertical_l...
2,Vertical Displacement,vertical_displacement_raw,vertical_displacement_calibrated,3662.900,29509.795097,µm,vertical_displacement_calibrated = 3662.9 × ve...


### Calculation sequence

The selected load and displacement have already been calibrated row by row:

$
F_{cal,i}=m_F F_{raw,i}+b_F
$

$
L_{cal,i}=m_L L_{raw,i}+b_L
$

The initial values are calculated as medians around the selected zero record. The zeroed axial force and measured shortening are:

$
F_i=F_{cal,i}-F_{cal,0}
$

$
\Delta L_{measured,i}=s\left(L_{cal,i}-L_{cal,0}\right)
$

where (s=+1) or (-1) defines the shortening sign. Calibrated load is used in kN and calibrated displacement in mm, so Step 4 applies no hidden conversion factors.

The initial area and engineering stress are:

$
A_0=\frac{\pi D_0^2}{4}
$

$
\sigma_1=\frac{F-F_0}{A_0}
$

with \(F\) in N and \(A_0\) in mm², giving stress directly in MPa.

Measured displacement includes deformation of both the specimen and loading system. If an independently measured machine stiffness \(k_m\) is available:

$
\Delta L_{corrected}=\Delta L_{recorded}-\frac{F-F_0}{k_m}
$

$
\varepsilon_a=\frac{\Delta L_{corrected}}{L_0}
$

Without a stiffness calibration, the notebook retains the zeroed displacement and labels it as uncorrected.

In [17]:
def median_around_record(series, record, half_window=5):
    start = max(0, int(record) - int(half_window))
    stop = min(len(series), int(record) + int(half_window) + 1)
    return float(np.nanmedian(np.asarray(series.iloc[start:stop], dtype=float)))


def reduce_complete_experiment(adc_frame, settings):
    required_columns = [settings['load_channel'], settings['displacement_channel']]
    missing = [column for column in required_columns if column not in adc_frame.columns]
    if missing:
        raise KeyError(f'Analysis channels were not found: {missing}')

    diameter = float(settings['diameter_mm'])
    length = float(settings['initial_length_mm'])
    if not np.isfinite(diameter) or diameter <= 0:
        raise ValueError('Enter a positive measured diameter_mm in ANALYSIS.')
    if not np.isfinite(length) or length <= 0:
        raise ValueError('Enter a positive measured initial_length_mm in ANALYSIS.')

    zero_record = int(settings['zero_record'])
    if not 0 <= zero_record < len(adc_frame):
        raise IndexError(f'zero_record must be between 0 and {len(adc_frame)-1}.')

    load = adc_frame[settings['load_channel']].astype(float)
    displacement = adc_frame[settings['displacement_channel']].astype(float)
    calibrated_units = {
        f'{safe_name(channel_name)}_calibrated': ADC_UNITS[channel_name]
        for channel_name in ADC_CHANNEL_NAMES
    }
    load_unit = calibrated_units[settings['load_channel']]
    displacement_unit = calibrated_units[settings['displacement_channel']]
    if load_unit.casefold() != 'kn':
        raise ValueError(
            f"The selected calibrated load channel must be in kN, not {load_unit!r}."
        )
    if displacement_unit not in ('µm', 'μm', 'um'):
        raise ValueError(
            'The selected calibrated displacement channel must be in µm, '
            f'not {displacement_unit!r}.'
        )

    half_window = int(settings['zero_half_window_records'])
    load_zero = median_around_record(load, zero_record, half_window)
    displacement_zero = median_around_record(displacement, zero_record, half_window)

    axial_force_kN = load - load_zero
    measured_shortening_mm = (
        settings['shortening_sign'] * (displacement - displacement_zero)/ 1000.0
    )
    stiffness = settings['machine_stiffness_kN_per_mm']
    if np.isfinite(stiffness):
        if stiffness <= 0:
            raise ValueError('machine_stiffness_kN_per_mm must be positive.')
        machine_displacement_mm = axial_force_kN / stiffness
        corrected_shortening_mm = measured_shortening_mm - machine_displacement_mm
        correction_status = 'machine-compliance corrected'
    else:
        machine_displacement_mm = pd.Series(np.nan, index=adc_frame.index)
        corrected_shortening_mm = measured_shortening_mm.copy()
        correction_status = 'uncorrected: machine stiffness not supplied'

    initial_area_mm2 = np.pi * diameter**2 / 4.0
    axial_stress_MPa = axial_force_kN * 1000.0 / initial_area_mm2
    axial_strain = corrected_shortening_mm / length

    # Preserve exactly one raw and one calibrated version of each ADC channel.
    # Calculated mechanical quantities are appended as separate analysis columns.
    reduced = adc_frame.copy()
    reduced['axial_force_zeroed_kN'] = axial_force_kN
    reduced['axial_stress_MPa'] = axial_stress_MPa
    reduced['axial_shortening_measured_mm'] = measured_shortening_mm
    reduced['machine_displacement_mm'] = machine_displacement_mm
    reduced['axial_shortening_corrected_mm'] = corrected_shortening_mm
    reduced['axial_strain'] = axial_strain
    reduced['axial_strain_percent'] = 100.0 * axial_strain

    audit = pd.DataFrame({
        'parameter': [
            'load_channel', 'load_unit', 'displacement_channel', 'displacement_unit',
            'zero_record', 'load_zero', 'displacement_zero', 'initial_area_mm2',
            'diameter_mm', 'initial_length_mm', 'machine_stiffness_kN_per_mm',
            'displacement_correction'
        ],
        'value': [
            settings['load_channel'], load_unit,
            settings['displacement_channel'], displacement_unit,
            zero_record, load_zero, displacement_zero, initial_area_mm2,
            diameter, length, stiffness, correction_status
        ]
    })
    return reduced, audit


# This cell intentionally stops until diameter and length have been entered above.
reduced_data, reduction_audit = reduce_complete_experiment(adc_data, ANALYSIS)
display(Markdown('### Reduction choices and zero values'))
display(reduction_audit)


### Reduction choices and zero values

,parameter,value
0,load_channel,vertical_load_calibrated
1,load_unit,kN
2,displacement_channel,vertical_displacement_calibrated
3,displacement_unit,µm
4,zero_record,2862
5,load_zero,1.322316
6,displacement_zero,17030.796241
7,initial_area_mm2,490.873852
8,diameter_mm,25.0
9,initial_length_mm,50.0


### ADC-channel retention check and reduced channels

The reduction must retain exactly one raw series and one calibrated series for every channel extracted from the ADC group. The first table verifies that both versions remain present and unchanged. This explicitly includes `AUX 5`, which will later synchronize mechanical and waveform data.

The second table lists every retained and calculated channel. Statistical summaries remain intentionally omitted.

In [18]:
retention_rows = []
for channel in adc_group.channels():
    base_name = safe_name(channel.name)
    raw_column = f'{base_name}_raw'
    calibrated_column = f'{base_name}_calibrated'
    raw_present = raw_column in reduced_data.columns
    calibrated_present = calibrated_column in reduced_data.columns
    unchanged = (
        raw_present and calibrated_present
        and np.allclose(reduced_data[raw_column], adc_data[raw_column], equal_nan=True)
        and np.allclose(reduced_data[calibrated_column], adc_data[calibrated_column], equal_nan=True)
    )
    retention_rows.append({
        'TDMS_channel': channel.name,
        'raw_column': raw_column,
        'calibrated_column': calibrated_column,
        'raw_present': raw_present,
        'calibrated_present': calibrated_present,
        'values_unchanged': unchanged,
    })

channel_retention = pd.DataFrame(retention_rows)
display(channel_retention)

if not channel_retention[['raw_present', 'calibrated_present', 'values_unchanged']].all().all():
    raise RuntimeError('At least one ADC channel was lost or changed during reduction.')

REDUCED_UNITS = {
    'record': '#',
    'elapsed_time_s': 's',
    'axial_force_zeroed_kN': 'kN',
    'axial_stress_MPa': 'MPa',
    'axial_shortening_measured_mm': 'mm',
    'machine_displacement_mm': 'mm',
    'axial_shortening_corrected_mm': 'mm',
    'axial_strain': '–',
    'axial_strain_percent': '%',
}

# Add units for every original ADC and calibrated channel.
for channel in adc_group.channels():
    base_name = safe_name(channel.name)
    REDUCED_UNITS[f'{base_name}_raw'] = 'raw ADC value'
    REDUCED_UNITS[f'{base_name}_calibrated'] = ADC_UNITS.get(channel.name, 'VERIFY')

reduced_inventory = pd.DataFrame({
    'channel': reduced_data.columns,
    'records': [len(reduced_data)] * len(reduced_data.columns),
    'unit': [REDUCED_UNITS.get(column, 'VERIFY') for column in reduced_data.columns],
})
display(reduced_inventory)


,TDMS_channel,raw_column,calibrated_column,raw_present,calibrated_present,values_unchanged
0,Vertical Load,vertical_load_raw,vertical_load_calibrated,True,True,True
1,Horizontal Load,horizontal_load_raw,horizontal_load_calibrated,True,True,True
2,Vertical Displacement,vertical_displacement_raw,vertical_displacement_calibrated,True,True,True
3,Horizontal Displacement,horizontal_displacement_raw,horizontal_displacement_calibrated,True,True,True
4,AUX 5,aux_5_raw,aux_5_calibrated,True,True,True
5,Time,time_raw,time_calibrated,True,True,True


,channel,records,unit
0,record,10740,#
1,vertical_load_raw,10740,raw ADC value
2,vertical_load_calibrated,10740,kN
3,horizontal_load_raw,10740,raw ADC value
4,horizontal_load_calibrated,10740,kN
5,vertical_displacement_raw,10740,raw ADC value
6,vertical_displacement_calibrated,10740,µm
7,horizontal_displacement_raw,10740,raw ADC value
8,horizontal_displacement_calibrated,10740,µm
9,aux_5_raw,10740,raw ADC value


## 5. Explore the reduced experiment

The same selector window is now applied to the reduced variables. Useful first comparisons include:

- axial stress versus elapsed time;
- corrected shortening versus elapsed time;
- axial stress versus axial strain;
- corrected versus measured shortening.

Do not divide the experiment into individual tests yet. First verify that signs, zeros, geometry and compliance correction produce a physically coherent complete record.

In [19]:
reduced_explorer = open_xy_explorer(
    reduced_data,
    units=REDUCED_UNITS,

    
    title='UCS experiment — reduced mechanical variables',
    include_raw=False,
)


In [20]:
# Export mechanical data for the acousto-mechanical waveform analyzer

required_columns = [
    'elapsed_time_s',
    'aux_5_calibrated',
]

missing = [
    column for column in required_columns
    if column not in reduced_data.columns
]

if missing:
    raise KeyError(
        f'Cannot create analyzer file. Missing columns: {missing}'
    )

# Rename the synchronization columns as expected by mech_wav_v2.py.
export_data = reduced_data.rename(columns={
    'elapsed_time_s': 'time_s',
    'aux_5_calibrated': 'sync',
}).copy()

# Place the two columns required by the analyzer first.
first_columns = ['time_s', 'sync']
remaining_columns = [
    column for column in export_data.columns
    if column not in first_columns
]
export_data = export_data[first_columns + remaining_columns]

# Build the units row expected as the second line of the file.
export_units = {
    'time_s': 's',
    'sync': 'V',
}

for column in remaining_columns:
    export_units[column] = REDUCED_UNITS.get(column, '–')

# Save beside the original TDMS file.
OUTPUT_PATH = TDMS_PATH.with_name(
    f'{TDMS_PATH.stem}_acousto_mech.txt'
)

with open(OUTPUT_PATH, 'w', encoding='utf-8', newline='') as output_file:
    # First row: column names
    output_file.write(
        '\t'.join(export_data.columns) + '\n'
    )

    # Second row: units; the analyzer skips this row
    output_file.write(
        '\t'.join(
            str(export_units.get(column, '–'))
            for column in export_data.columns
        ) + '\n'
    )

    # Remaining rows: numerical data
    export_data.to_csv(
        output_file,
        sep='\t',
        index=False,
        header=False,
        float_format='%.10g',
    )

print('Analyzer-ready file created:')
print(OUTPUT_PATH.resolve())
print(f'Rows exported: {len(export_data):,}')
print(f'Columns exported: {len(export_data.columns)}')
print("Required analyzer columns: 'time_s' and 'sync'")

Analyzer-ready file created:
/Users/marcoscuderi/Library/CloudStorage/GoogleDrive-marco.scuderi@uniroma1.it/My Drive/Comunications/Classes/04_petrophysics/L07_part_a_b/UCS/calibration/triax_calibration_acousto_mech.txt
Rows exported: 10,740
Columns exported: 21
Required analyzer columns: 'time_s' and 'sync'


## Stop point for testing

Before continuing to test segmentation or acoustic analysis, check:

1. Are all startup PID values unpacked correctly?
2. Do the calibration coefficients agree with the apparatus configuration?
3. Are channel units correct?
4. Does the pop-up explorer work in the course Anaconda environment?
5. Is the chosen zero record defensible?
6. Does corrected shortening have the expected sign?
7. Does the stress–strain response remain physically plausible over the complete record?

Once these points are confirmed, the next version can add test separation, property extraction and synchronized acoustic analysis.